In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, regexp_extract, url_decode, when, lit, udf
from pyspark.sql.types import StringType
import geoip2.database

reader = geoip2.database.Reader('/src/data/GeoLite2-City.mmdb')

def get_city(ip):
    try:
        response = reader.city(ip)
        return response.city.name
    except:
        return None

def get_country(ip):
    try:
        response = reader.city(ip)
        return response.country.name
    except:
        return None
get_city_udf = udf(get_city, StringType())

In [13]:
spark  = SparkSession.builder.appName("no").getOrCreate()

In [14]:
get_city_udf("128.101.101.101")

Traceback (most recent call last):
  File "/opt/spark/python/pyspark/serializers.py", line 459, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
  File "/opt/spark/python/pyspark/cloudpickle/cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "/opt/spark/python/pyspark/cloudpickle/cloudpickle_fast.py", line 632, in dump
    return Pickler.dump(self, obj)
TypeError: cannot pickle 'Reader' object


PicklingError: Could not serialize object: TypeError: cannot pickle 'Reader' object

In [3]:
spark.sql("SHOW DATABASES IN lakehouse").show()


+---------+
|namespace|
+---------+
|   bronze|
+---------+



In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS bronze")

DataFrame[]

In [8]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_clickstream (
        ts TIMESTAMP_NTZ,
        ip STRING,
        url STRING
    )
    USING iceberg
    LOCATION 's3://lakehouse/bronze/raw_clickstream'
""")


DataFrame[]

In [9]:
df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/src/data/logs_2017.csv")

df.show(5)  # Hiển thị 5 dòng đầu
df.printSchema()  # Xem schema của DataFrame


+-------------------+--------------+--------------------+
|                 ts|            ip|                 url|
+-------------------+--------------+--------------------+
|2017-09-01 06:00:41| 215.143.180.0|/department/appar...|
|2017-09-01 06:00:43|  206.56.112.1|/department/footw...|
|2017-09-01 06:00:49|  37.97.182.65|/department/fitne...|
|2017-09-01 06:00:50|  206.56.112.1|/department/fan%2...|
|2017-09-01 06:01:39|136.108.56.242|/department/fan%2...|
+-------------------+--------------+--------------------+
only showing top 5 rows

root
 |-- ts: timestamp (nullable = true)
 |-- ip: string (nullable = true)
 |-- url: string (nullable = true)



In [10]:
df.write.format("iceberg") \
    .mode("overwrite") \
    .save("lakehouse.bronze.raw_clickstream")
